In [ ]:
import requests
from bs4 import BeautifulSoup, Comment
import pandas as pd
import time
from unidecode import unidecode
import numpy as np
from itertools import chain


#make sure do not go over rate limit of 20 requests per minute. If over, sportsreference puts in "jail" for an hour
rate = 0
def checkRate(r):
    global rate
    time.sleep(5)

#since roster table is hidden in comments, needs this to fish it out
def makeCommentTable(soup1, type):
    global rate

    #finds all comments in soup and makes them not comments and just html. 
    comments = soup1.find_all(string=lambda text: isinstance(text, Comment) and "table_container" in text)
    html = str(comments).replace('<!--', '').replace('-->', '')
    
    #parse through the HTML content using Beautiful Soup
    soup = BeautifulSoup(html, 'html.parser')

    #reads roster table html into a dataframe
    if type == False:
        #find the roster table by its HTML id
        rosterTable = soup.find('table', {'id': 'roster'})
        df = pd.read_html(str(rosterTable))[0]
    else:
        passingTable = soup.find('table', {'id': 'passing'})
        otherTable = soup.find('table', {'id': 'rushing_and_receiving'})

        #make array of two df instead of one, then flatten it to 1d array to be used later
        df = [pd.read_html(str(passingTable)), pd.read_html(str(otherTable))]
        df = list(chain.from_iterable(df))

    return df

def rosterMaker():

    #empty df to add things to
    df = pd.DataFrame(columns= ["No.", "Player", "Age", "Pos", "G", "GS", "Wt", "Ht", "College/Univ", "BirthDate", "Yrs", "AV", "Drafted (tm/rnd/yr)", "Year", "YearsBack", "Team"])

    global rate

    #years used for grading. changes each season. if making currYearRoster, use array below with just current year
    #years = ["2022", "2021", "2020", "2019"]
    years = ["2013","2014","2015","2016","2017","2018","2019","2020","2021","2022","2023"]

    #team abbr list
    teams = ["crd", "atl", "rav", "buf", "car", "chi", "cin", "cle", "dal", "den", "det", "gnb", "htx", "clt", "jax", "kan", "rai", "sdg", "ram", "mia", "min", "nwe", "nor", "nyg", "nyj", "phi", "pit", "sfo", "sea", "tam", "oti", "was"]

    #used to find yearsback from present
    x = 1

    #loop to go through each team in each year and make df of roster for each team per year
    for num in years:
        for item in teams: 
            #makes url for every team
            url = "https://www.pro-football-reference.com/teams/" + item + "/" + num + "_roster.htm"

            #get page wanted and make a beautiful soup out of it.
            checkRate(rate)
            response = requests.get(url)
            print(response)
            rate += 1

            soup = BeautifulSoup(response.text, 'html.parser')

            #dataframe of table
            table = makeCommentTable(soup, False)

            #add in year and years back columns
            table['Year'] = num
            table["YearsBack"] = x
            table["Team"] = item


            #make all into one df
            df = pd.concat([df, table], ignore_index=True, join="inner")
                
        x+=1
        
        
    
    #write dfs into csv files for later use
    if len(years)==1:
        #change all rb type positions to rb
        rbValues = ['RB', 'HB', 'TB', 'FB', "LH", "RH", "BB", "B", "WB"]
        df.loc[df['Pos'].isin(rbValues), 'Pos'] = "RB"

        #make only first two strings kept, and only keep rb, wr, qb, te columns
        df.loc[df['Pos'].str.len() > 2, 'Pos'] = df.loc[df['Pos'].str.len() > 2, 'Pos'].apply(lambda x: x[:2])
        keepers = ["RB", "QB", "TE", "WR"]

        df = df.loc[df['Pos'].isin(keepers)]

        df.to_pickle("currYearRoster.pkl")
    else:
        df.to_pickle("teamsPastRoster.pkl")

rosterMaker()

<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200

In [1]:
import pandas as pd
teamsPast = pd.read_pickle("teamsPastRoster.pkl")
teamsPast['YearsBack'] = 2024 - teamsPast['Year'].astype(int)
teamsPast.to_pickle('/Users/kmaran3/Dropbox/Darkhorse/teamsPastRoster.pkl')

ImportError: Unable to import required dependencies:
numpy: 

IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!

Importing the numpy C-extensions failed. This error can happen for
many reasons, often due to issues with your setup or how NumPy was
installed.

We have compiled some common reasons and troubleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python3.10 from "/Users/kmaran3/anaconda3/bin/python"
  * The NumPy version is: "1.23.5"

and make sure that they are the versions you expect.
Please carefully study the documentation linked above for further help.

Original error was: dlopen(/Users/kmaran3/anaconda3/lib/python3.10/site-packages/numpy/core/_multiarray_umath.cpython-310-darwin.so, 0x0002): Library not loaded: @rpath/libopenblas.0.dylib
  Referenced from: <565CCDA0-1598-30EB-9212-737683B9FED1> /Users/kmaran3/anaconda3/lib/python3.10/site-packages/numpy/

In [ ]:
df